# 1. Fundamentos de Agentes de IA

## Objetivos de Aprendizaje
- Comprender qué es un agente de IA y cuáles son sus componentes clave.
- Entender el ciclo de razonamiento (ReAct) que sigue un agente para resolver problemas.
- Implementar un agente simple desde cero en Python (Vanilla Code).
- Reconocer las limitaciones de un agente básico y la necesidad de frameworks como LangChain o CrewAI.

## ¿Qué es un Agente de IA?

Un agente de IA no es simplemente un modelo de lenguaje (LLM) que responde a preguntas. Es un sistema más avanzado que utiliza un LLM como su **cerebro (core engine)** para razonar y tomar decisiones. A diferencia de una simple llamada a una API, un agente puede:

1.  **Descomponer un objetivo complejo** en una secuencia de pasos intermedios.
2.  **Interactuar con herramientas externas** (APIs, bases de datos, funciones de código) para obtener información o ejecutar acciones en el mundo real.
3.  **Observar los resultados** de esas acciones y ajustar su plan en consecuencia.
4.  **Repetir este ciclo** hasta que el objetivo original se haya cumplido.

Piénsalo como un becario inteligente: le das una tarea de alto nivel (ej. "Investiga el precio de las acciones de Apple y dime si es un buen momento para comprar"), y él solo descubre qué herramientas usar (búsqueda web, una API financiera), cómo usarlas y cómo interpretar los resultados para darte una recomendación.

### Componentes Clave de un Agente

Un agente, en su forma más básica, se compone de tres elementos principales:

1.  **Cerebro (Core Engine)**: El LLM que impulsa al agente. Es responsable del razonamiento, la planificación y la toma de decisiones.
2.  **Memoria (Memory)**: Un sistema para almacenar y recuperar información de la conversación actual (memoria a corto plazo) o de interacciones pasadas (memoria a largo plazo). Esto le da contexto al agente.
3.  **Herramientas (Tools)**: Funciones o APIs que el agente puede "llamar" para interactuar con el mundo exterior. Esto supera la limitación del conocimiento estático del LLM.

## Implementación de un Agente Básico en Python

In [1]:
!pip install openai python-dotenv -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime # hora y fecha actual

load_dotenv()  # Carga las variables de entorno desde el archivo .env

# --- 1. Configuración del Cliente OpenAI (vía Groq) ---
# Asegúrate de tener las variables de entorno GROQ_BASE_URL y GROQ_API_KEY configuradas
try:
    client = OpenAI(
        base_url=os.environ.get("GROQ_BASE_URL"),
        api_key=os.environ.get("GROQ_API_KEY")
    )
    print("✅ Cliente OpenAI configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando el cliente: {e}")
    client = None

✅ Cliente OpenAI configurado correctamente.


### 2. Definición de las Herramientas (Tools)

Vamos a crear dos herramientas muy simples que nuestro agente podrá usar:

In [3]:
def get_current_time():
    """Devuelve la fecha y hora actual."""
    return f"La fecha y hora actual es: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def search_web(query):
    """Simula una búsqueda web para un término dado."""
    # En una implementación real, aquí llamaríamos a una API de búsqueda (ej. Google, Bing)
    print(f"🔎 Buscando en la web: '{query}'...")
    if "elon musk" in query.lower():
        return "Elon Musk es el CEO de SpaceX y Tesla."
    elif "inteligencia artificial" in query.lower():
        return "La IA es un campo de la informática dedicado a crear sistemas que pueden realizar tareas que normalmente requieren inteligencia humana."
    else:
        return f"No se encontraron resultados para '{query}'."

# Mapeo de herramientas para que el agente sepa qué funciones puede llamar (contiene 2 claves)
tools = {
    "get_current_time": {
        "function": get_current_time,
        "description": "Útil para obtener la fecha y hora actual.",
        "args": {}
    },
    "search_web": {
        "function": search_web,
        "description": "Útil para buscar información en internet sobre personas, lugares o conceptos.",
        "args": {"query": "la pregunta a buscar"}
    }
}

print("✅ Herramientas del agente definidas.")

✅ Herramientas del agente definidas.


In [ ]:
# ---------- PARTE 1: armar el prompt del sistema ----------
def create_system_prompt(tools): 
    prompt = """Eres un asistente útil que puede usar herramientas para responder preguntas. Sigue estrictamente el siguiente formato:

**Thought (Pensamiento):** El razonamiento sobre qué hacer a continuación.
**Action (Acción):** La herramienta a usar, en formato JSON. Debe ser una de las siguientes: {tool_names}
**Observation (Observación):** El resultado de la acción.
**Final Answer (Respuesta Final):** La respuesta final a la pregunta original.

Para responder, debes seguir este ciclo de Pensamiento -> Acción -> Observación tantas veces como sea necesario. Cuando tengas la respuesta final, usa el formato 'Final Answer'."""

    # 1.1 Una frase de texto por cada herramienta (nombre + descripción + argumentos)
    tool_descs = []
    for name, details in tools.items():
        tool_descs.append(f"- {name}: {details['description']} Argumentos: {details['args']}")

    # 1.2 Rellena {tool_names} y pega las descripciones al final: todo queda en un solo texto
    return prompt.format(tool_names=json.dumps(list(tools.keys()))) + "Herramientas disponibles:" + "".join(tool_descs)


# ---------- PARTE 2: el agente ----------
def run_agent(user_query, client, tools):
    # 2.1 Si el cliente no se creó, se sale de la función
    if not client:
        print("❌ Cliente no inicializado.")
        return

    # 2.2 Se arma el prompt del sistema como contenido del del mensaje para el modelo
    system_prompt = create_system_prompt(tools) # devuelve un texto con las instrucciones y la lista de herramientas
    print(system_prompt)
    messages = [
        {"role": "system", "content": system_prompt},  # instrucciones
        {"role": "user", "content": user_query}        # pregunta del usuario
    ]

    print(f"--- Agente iniciado para la consulta: '{user_query}' ---")

    # 2.3 Ciclo del agente: máximo 5 vueltas para evitar bucles infinitos
    for _ in range(5):

        # 2.4 Se le manda la conversación al modelo
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages, # se le manda la coversación al modelo (instrucciones + pregunta + observaciones previas)
            temperature=0,
            max_tokens=2000
        )

        # 2.5 Se saca el texto de la respuesta y se guarda en la conversación
        text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(f"🤖 Pensamiento del Agente:{text}")

        # 2.6 Si dice "Final Answer:" : devuelve solo lo que viene después
        if "Final Answer:" in text:
            final_answer = text.split("Final Answer:")[-1].strip()
            print(f"--- ✅ Agente ha finalizado --- ")
            return final_answer

        # 2.7 Si no terminó, busca si el modelo pidió una herramienta ("Action: {"tool": ..., "args": {...}}")
        action_match = re.search(r"Action: (\{.*?\})", text, re.DOTALL)
        if action_match:
            try:
                # 2.8 Convierte el JSON en diccionario y saca herramienta + argumentos
                action_json = json.loads(action_match.group(1).strip())
                tool_name = action_json["tool"]
                tool_args = action_json["args"]

                if tool_name in tools:
                    # 2.9 Ejecuta la herramienta y le devuelve el resultado como texto para enviar al modelo
                    observation = tools[tool_name]["function"](tool_args)
                    observation_text = f"Observation: {observation}"
                    messages.append({"role": "user", "content": observation_text})
                else:
                    # 2.10 La herramienta no existe: se le avisa al modelo
                    messages.append({"role": "user", "content": f"Observation: Herramienta '{tool_name}' desconocida."})
            except Exception as e:
                # 2.11 Si algo falló, se le avisa al modelo en vez de detener el agente
                messages.append({"role": "user", "content": f"Observation: Error al ejecutar la acción - {str(e)}"})
        else:
            # 2.12 Ni Final Answer ni Action: el agente se detiene
            print("--- ⚠️  El agente no pudo determinar una acción y se detuvo. ---")
            return text

    # 2.13 Si se acaban las 5 vueltas sin Final Answer, llega aquí
    print("--- 🛑 Límite de iteraciones alcanzado. ---")
    return "El agente no pudo completar la tarea en el número máximo de pasos."

print("✅ Lógica del agente definida.")

✅ Lógica del agente definida.


In [5]:
tool_descs = []

for name, details in tools.items():
    print("name    =", name)
    print("details =", details)
    linea = f"- {name}: {details['description']} Argumentos: {details['args']}"
    print("linea   =", linea)
    print("-----")
    tool_descs.append(linea)

print("Lista final:")
print(tool_descs)


name    = get_current_time
details = {'function': <function get_current_time at 0x000002D30852B1C0>, 'description': 'Útil para obtener la fecha y hora actual.', 'args': {}}
linea   = - get_current_time: Útil para obtener la fecha y hora actual. Argumentos: {}
-----
name    = search_web
details = {'function': <function search_web at 0x000002D30864A350>, 'description': 'Útil para buscar información en internet sobre personas, lugares o conceptos.', 'args': {'query': 'la pregunta a buscar'}}
linea   = - search_web: Útil para buscar información en internet sobre personas, lugares o conceptos. Argumentos: {'query': 'la pregunta a buscar'}
-----
Lista final:
['- get_current_time: Útil para obtener la fecha y hora actual. Argumentos: {}', "- search_web: Útil para buscar información en internet sobre personas, lugares o conceptos. Argumentos: {'query': 'la pregunta a buscar'}"]


### 4. Ejecución del Agente

Ahora, pongamos a nuestro agente a trabajar con una pregunta que requiere usar una herramienta.

In [6]:
import json
final_response = run_agent("¿Quién es Elon Musk y qué hora es?", client, tools)
print(f"🏁 Respuesta Final del Agente: {final_response}")

Eres un asistente útil que puede usar herramientas para responder preguntas. Sigue estrictamente el siguiente formato:

**Thought (Pensamiento):** El razonamiento sobre qué hacer a continuación.
**Action (Acción):** La herramienta a usar, en formato JSON. Debe ser una de las siguientes: ["get_current_time", "search_web"]
**Observation (Observación):** El resultado de la acción.
**Final Answer (Respuesta Final):** La respuesta final a la pregunta original.

Para responder, debes seguir este ciclo de Pensamiento -> Acción -> Observación tantas veces como sea necesario. Cuando tengas la respuesta final, usa el formato 'Final Answer'.Herramientas disponibles:- get_current_time: Útil para obtener la fecha y hora actual. Argumentos: {}- search_web: Útil para buscar información en internet sobre personas, lugares o conceptos. Argumentos: {'query': 'la pregunta a buscar'}
--- Agente iniciado para la consulta: '¿Quién es Elon Musk y qué hora es?' ---
🤖 Pensamiento del Agente:**Final Answer:** E

## Conclusiones y Próximos Pasos

Hemos construido un agente funcional desde cero. Sin embargo, hemos tenido que manejar mucha lógica compleja:

- **Análisis de la respuesta del LLM**: Usar expresiones regulares (`re`) y `json.loads` para extraer la acción es frágil y propenso a errores.
- **Gestión del prompt**: Construir y actualizar el prompt manualmente es tedioso.
- **Manejo del ciclo**: El bucle `for` con la lógica de parada es repetitivo.
- **Escalabilidad**: Añadir más herramientas, gestionar la memoria o implementar planes más complejos se volvería muy difícil.

**Aquí es donde entran los frameworks como LangChain y CrewAI.**

Estos frameworks abstraen toda esta complejidad, permitiéndonos definir agentes, herramientas y tareas de una manera mucho más declarativa y robusta. En los próximos notebooks, veremos cómo recrear este mismo agente usando estas herramientas para apreciar la diferencia en simplicidad y potencia.